In [1]:
import ast
import concurrent.futures
import glob
import itertools
import os
import pickle
import warnings
import sys

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.dates import YearLocator


import numpy as np
import pandas as pd
import polars as pl
import statsmodels.api as sm

from concurrent.futures import ThreadPoolExecutor
from joblib import Parallel, delayed
from multiprocessing import Pool, cpu_count

from sklearn.linear_model import LinearRegression
from sklearn.metrics import pairwise_distances, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split, cross_val_score
#from sklearn.cluster import KMeans

from statsmodels.regression.rolling import RollingOLS

from tqdm.notebook import tqdm
from collections import Counter
from functools import reduce
from pprint import pprint

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.6f}'.format)


warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

### Read Blocks

In [2]:
# Directory where the files are located
block_dir = "../../data/imputed_block_windowsize=2"

# Function to extract the number from the filename
def get_block_number(filename):
    return int(filename.split('_')[1].split('.')[0])

# Function to read a CSV file
def read_csv_file(filename):
    return pd.read_csv(filename)

# Function to read files in parallel and store them in an array
def read_files_in_parallel(file_list):
    results = []
    with ThreadPoolExecutor() as executor:
        # Use map to apply the read_csv_file function to each file in parallel
        results = list(executor.map(lambda f: read_csv_file(os.path.join(block_dir, f)), file_list))
    return results

In [3]:
if not (os.path.exists("imputed_even_blocks.parquet") and os.path.exists("imputed_odd_blocks.parquet")):

    # Get all filenames that follow the naming convention
    all_files = [f for f in os.listdir(block_dir) if f.startswith('block_') and f.endswith('.csv')]
    
    # Separate the filenames into even and odd block numbers
    even_files = [f for f in all_files if get_block_number(f) % 2 == 0]
    odd_files = [f for f in all_files if get_block_number(f) % 2 != 0]
    
    # Read the even and odd files in parallel
    even_dataframes = read_files_in_parallel(even_files)
    odd_dataframes = read_files_in_parallel(odd_files)
    
    # Combine each array of DataFrames into a single DataFrame using pandas.concat
    even_combined_df = pd.concat(even_dataframes, ignore_index=True).sort_values(["fips","datetime"])
    odd_combined_df = pd.concat(odd_dataframes, ignore_index=True).sort_values(["fips","datetime"])
    
    print("Writing parquet")
    even_combined_df.to_parquet("imputed_even_blocks.parquet", index=False)
    odd_combined_df.to_parquet("imputed_odd_blocks.parquet", index=False)
else:
    even_combined_df = pl.read_parquet("imputed_even_blocks.parquet")
    odd_combined_df = pl.read_parquet("imputed_odd_blocks.parquet")
           

In [4]:
numeric_even_combined_df = even_combined_df.select(
    pl.col(pl.Float64, pl.Float32, pl.Int64, pl.Int32)
)
numeric_odd_combined_df = odd_combined_df.select(
    pl.col(pl.Float64, pl.Float32, pl.Int64, pl.Int32)
)


In [5]:
# Columns to exclude
index_cols = ["fips", "cutoff"]
exclude_cols = ["State_FIPS_Code", "log_rolled_cases.x", "log_rolled_cases.y", "t0.lm", "r.lm"]
outcome_col = "shifted_log_rolled_cases"
feature_cols = list(set(numeric_even_combined_df.columns) - set(index_cols) - set(exclude_cols) - set(outcome_col))
print(feature_cols)

['Variant % BN.1', 'Stay_at_home/_shelter_in_place', 'Face_mask_mandate_enforced_by_fines', 'EPL_AGE17', 'metrics.testPositivityRatio', 'Date_adults_ages_70+_became_eligible_for_COVID-19_vaccination', 'MP_NOVEH', 'Report_COVID-19_testing_by_race/ethnicity', 'COVID-19_is_not_an_acceptable_reason_to_request_application_for_mail-in_ballot_unless_sick_or_exposed_(as_of_September_16,_2020)', 'July_2020_unemployment_insurance_maximum_duration_(weeks)', 'Variant % BF.7', 'Reopened_other_non-essential_retail', 'E_POV', 'Closed_restaurants_(x2)', 'Reopen_restaurants', 'No_state_unemployment_waiting_period_prior_to_pandemic;_or_date_waiting_period_waived_not_found', 'totalTestEncountersViralIncrease', 'Variant % B.1.427', 'totalTestsPeopleAntibody', 'Face_mask_mandate_x2', 'Date_banned_visitors_to_nursing_homes', 'F_CROWD', 'Reopened_casinos_(x2)', 'State_of_emergency_issued', 'onVentilatorCurrently', 'EP_DISABL', 'Closed_casinos_(x2)', 'RPL_THEMES', 'Population_density_per_square_miles', 'Proof

In [6]:
scaled_numeric_even_combined_df = numeric_even_combined_df.with_columns([
    pl.when(pl.col(col).std() != 0)  # Check if standard deviation is not zero
    .then((pl.col(col) - pl.col(col).mean()) / pl.col(col).std())  # Standardize
    .otherwise(pl.col(col))  # Keep the column unchanged if constant
    .alias(col)
    for col in feature_cols
])

scaled_numeric_odd_combined_df = numeric_odd_combined_df.with_columns([
    pl.when(pl.col(col).std() != 0)  # Check if standard deviation is not zero
    .then((pl.col(col) - pl.col(col).mean()) / pl.col(col).std())  # Standardize
    .otherwise(pl.col(col))  # Keep the column unchanged if constant
    .alias(col)
    for col in feature_cols
])


In [7]:
even_nan_mask = numeric_even_combined_df.select([pl.col(col).is_nan().alias(f"{col}_is_nan") for col in numeric_even_combined_df.columns])
odd_nan_mask = numeric_odd_combined_df.select([pl.col(col).is_nan().alias(f"{col}_is_nan") for col in numeric_odd_combined_df.columns])


In [8]:
even_nan_mask.sum().to_pandas().values

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [9]:
odd_nan_mask.sum().to_pandas().values

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [10]:
scaled_numeric_even_combined_df.write_parquet("scaled_numeric_even_combined_df.parquet")
scaled_numeric_odd_combined_df.write_parquet("scaled_numeric_odd_combined_df.parquet")